# ENERGICAL — Time Series Analysis & Revenue Forecast by Wilaya
### DataFest 2026 | The Outliers | June 2026

## 1 · Data Loading & Wilaya Selection

Reads from `df_clean.csv` (output of `01_EDA_cleaning.ipynb`).
User selects a wilaya interactively — the rest of the notebook runs on that wilaya only

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

transactions = pd.read_csv('data_clean.csv', parse_dates=['date_commande'])

print("\nAvailable Wilayas:\n")

for w in sorted(transactions["wilaya"].unique()):
    print("-", w)


In [ ]:

selected = input("\nEnter Wilaya exactly as written: ").strip()

filtered = transactions[
    transactions["wilaya"] == selected
]

if filtered.empty:
    raise ValueError("No data found for selected wilaya.")



In [ ]:

series = (
    filtered
    .groupby(
        pd.Grouper(
            key="date_commande",
            freq="ME"
        )
    )["montant_da"]
    .sum()
)

series = series.asfreq("ME", fill_value=0)

series.name = "Revenue"


plt.figure(figsize=(13,6))

plt.plot(
    series.index,
    series.values,
    marker="o",
    linewidth=2
)

plt.title(f"Monthly Revenue - {selected}")
plt.xlabel("Date")
plt.ylabel("Revenue (DA)")
plt.grid(True)
plt.tight_layout()
plt.show()



print("\nNumber of observations:", len(series))

print("\nDate range:")
print(series.index.min())
print(series.index.max())

## 2 · Exploratory Time Series Analysis

Train/test split: last 3 months held out as test set.
We analyze stationarity (ADF test) and volatility (coefficient of variation) to understand the series before modeling.

In [ ]:

from statsmodels.tsa.stattools import adfuller

forecast_horizon = 3

train = series.iloc[:-forecast_horizon]
test = series.iloc[-forecast_horizon:]

print("\n========================================")
print("TRAIN / TEST SPLIT")
print("========================================")

print(f"Training observations : {len(train)}")
print(f"Testing observations  : {len(test)}")



In [ ]:

plt.figure(figsize=(13,6))

plt.plot(
    train.index,
    train.values,
    marker="o",
    linewidth=2,
    label="Training Data"
)

plt.title("Training Time Series")

plt.xlabel("Date")
plt.ylabel("Revenue (DA)")

plt.grid(True)

plt.legend()

plt.tight_layout()

plt.show()

print("\n========================================")
print("DESCRIPTIVE STATISTICS")
print("========================================")

print(train.describe())

cv = train.std() / train.mean() * 100

print(f"\nMean Revenue               : {train.mean():.2f}")
print(f"Median Revenue             : {train.median():.2f}")
print(f"Standard Deviation         : {train.std():.2f}")
print(f"Variance                   : {train.var():.2f}")
print(f"Minimum                    : {train.min():.2f}")
print(f"Maximum                    : {train.max():.2f}")
print(f"Coefficient of Variation   : {cv:.2f}%")



print("\n========================================")
print("ADF TEST")
print("========================================")

adf = adfuller(train)

print(f"ADF Statistic : {adf[0]:.4f}")
print(f"P-value       : {adf[1]:.4f}")

print("\nCritical Values:")

for key, value in adf[4].items():
    print(f"{key}: {value:.4f}")

stationary = adf[1] < 0.05

print("\nInterpretation:")

if stationary:
    print("The series appears stationary.")
else:
    print("The series appears non-stationary.")
    print("Auto ARIMA will automatically determine the required differencing.")



In [ ]:

print("\n========================================")
print("BUSINESS SUMMARY")
print("========================================")

if cv < 20:
    print("Revenue is relatively stable over time.")

elif cv < 50:
    print("Revenue shows moderate variability.")

else:
    print("Revenue is highly variable.")

if stationary:
    print("Historical revenue is statistically stable.")
else:
    print("Historical revenue changes over time.")

## 3 · Model Comparison

In [ ]:

from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    root_mean_squared_error
)

from pmdarima import auto_arima
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

seasonal_period = 12

results = {}
predictions = {}
models = {}


def evaluate(actual, predicted):

    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": root_mean_squared_error(actual, predicted),
        "MAPE": mean_absolute_percentage_error(actual, predicted)
    }



print("Fitting Auto ARIMA...")

try:

    arima = auto_arima(
        train,
        seasonal=True,
        m=12,
        trace=False,
        stepwise=True,
        suppress_warnings=True
    )

    pred = pd.Series(
        arima.predict(n_periods=len(test)),
        index=test.index
    )

    models["Auto ARIMA"] = arima
    predictions["Auto ARIMA"] = pred
    results["Auto ARIMA"] = evaluate(test, pred)

except Exception as e:

    print(e)


print("Fitting Holt-Winters...")

try:

    hw = ExponentialSmoothing(
        train,
        trend="add",
        seasonal="add",
        seasonal_periods=12
    ).fit()

    pred = hw.forecast(len(test))

    models["Holt-Winters"] = hw
    predictions["Holt-Winters"] = pred
    results["Holt-Winters"] = evaluate(test, pred)

except Exception as e:

    print(e)


print("Fitting ETS...")

try:

    ets = ETSModel(
        train,
        error="add",
        trend="add",
        seasonal="add",
        seasonal_periods=12
    ).fit()

    pred = ets.forecast(len(test))

    models["ETS"] = ets
    predictions["ETS"] = pred
    results["ETS"] = evaluate(test, pred)

except Exception as e:

    print(e)


comparison = pd.DataFrame(results).T

comparison = comparison.sort_values("RMSE")

print("\n==============================")
print("MODEL COMPARISON")
print("==============================")

print(comparison.round(2))

best_model_name = comparison.index[0]

print("\nBest Model:")

print(best_model_name)

In [ ]:


# ==========================================
# PART 4 - FINAL FORECAST WITH 95% CI
# ==========================================

forecast_steps = 3

print("\n========================================")
print("TRAINING BEST MODEL ON FULL DATA")
print("========================================")

z = norm.ppf(0.975)   # 95% confidence level

# ==========================================
# FIT BEST MODEL
# ==========================================

if best_model_name == "Auto ARIMA":

    final_model = auto_arima(
        series,
        seasonal=True,
        m=12,
        trace=False,
        stepwise=True,
        suppress_warnings=True
    )

    forecast, conf_int = final_model.predict(
        n_periods=forecast_steps,
        return_conf_int=True,
        alpha=0.05
    )

    lower = conf_int[:, 0]
    upper = conf_int[:, 1]

elif best_model_name == "Holt-Winters":

    final_model = ExponentialSmoothing(
        series,
        trend="add",
        seasonal="add",
        seasonal_periods=12
    ).fit()

    forecast = final_model.forecast(forecast_steps).values

    sigma = np.std(final_model.resid, ddof=1)

    lower = forecast - z * sigma
    upper = forecast + z * sigma

elif best_model_name == "ETS":

    final_model = ETSModel(
        series,
        error="add",
        trend="add",
        seasonal="add",
        seasonal_periods=12
    ).fit()

    forecast = final_model.forecast(forecast_steps).values

    sigma = np.std(final_model.resid, ddof=1)

    lower = forecast - z * sigma
    upper = forecast + z * sigma

# ==========================================
# CREATE FORECAST TABLE
# ==========================================

future_dates = pd.date_range(
    start=series.index[-1] + pd.offsets.MonthEnd(1),
    periods=forecast_steps,
    freq="ME"
)

forecast_df = pd.DataFrame({
    "Forecast Revenue": forecast,
    "Lower 95% CI": lower,
    "Upper 95% CI": upper
}, index=future_dates)

print("\n========================================")
print("FORECAST WITH 95% CONFIDENCE INTERVAL")
print("========================================")

print(forecast_df.round(2))

# ==========================================
# PLOT
# ==========================================

plt.figure(figsize=(14,6))

plt.plot(
    series.index,
    series.values,
    marker="o",
    linewidth=2,
    label="Historical Revenue"
)

plt.plot(
    forecast_df.index,
    forecast_df["Forecast Revenue"],
    marker="o",
    linestyle="--",
    linewidth=3,
    label="Forecast"
)

plt.fill_between(
    forecast_df.index,
    forecast_df["Lower 95% CI"],
    forecast_df["Upper 95% CI"],
    alpha=0.25,
    label="95% Confidence Interval"
)

plt.title(f"{best_model_name} Revenue Forecast")
plt.xlabel("Date")
plt.ylabel("Revenue (DA)")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

### 4 · Final Forecast table

In [ ]:

future_dates = pd.date_range(
    start=series.index[-1] + pd.offsets.MonthEnd(1),
    periods=forecast_steps,
    freq="ME"
)

forecast = pd.Series(
    forecast,
    index=future_dates,
    name="Forecast Revenue"
)

print("\n========================================")
print("FORECAST")
print("========================================")

print(forecast.round(2))


plt.figure(figsize=(14,6))

plt.plot(
    series.index,
    series.values,
    marker="o",
    linewidth=2,
    label="Historical Revenue"
)

plt.plot(
    forecast.index,
    forecast.values,
    marker="o",
    linewidth=3,
    linestyle="--",
    label="Forecast"
)

plt.title("Revenue Forecast")

plt.xlabel("Date")
plt.ylabel("Revenue (DA)")

plt.grid(True)

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
forecast_table = pd.DataFrame({
    "Month": forecast.index.strftime("%B %Y"),
    "Forecast Revenue (DA)": forecast.round(2)
})

print(forecast_table.to_string(index=False))

## 5 · Business Decision Report

In [ ]:

import numpy as np

historical_mean = series.mean()
forecast_mean = forecast.mean()

change_ratio = (forecast_mean - historical_mean) / historical_mean

# simple trend direction
trend_value = np.polyfit(range(len(series)), series.values, 1)[0]

# volatility proxy
volatility = series.std() / series.mean()

forecast_volatility = forecast.std() / forecast.mean()



## 6. BUSINESS INTERPRETATION

In [ ]:

# TREND
if trend_value > 0:
    outlook = "Revenue is expected to grow"
elif trend_value < 0:
    outlook = "Revenue is expected to decline"
else:
    outlook = "Revenue is expected to remain stable"

# DEMAND STRENGTH
if change_ratio > 0.1:
    demand = "Strong increase in demand expected"
elif change_ratio < -0.1:
    demand = "Decrease in demand expected"
else:
    demand = "Stable demand expected"

# RISK LEVEL (business risk, not statistical)
if forecast_volatility < 0.2:
    risk = "LOW"
elif forecast_volatility < 0.5:
    risk = "MEDIUM"
else:
    risk = "HIGH"

# INVENTORY ACTION
if demand == "Strong increase in demand expected":
    action = "Increase stock levels and secure additional supply"

elif demand == "Decrease in demand expected":
    action = "Reduce stock purchases and limit new inventory"

else:
    action = "Maintain current stock strategy"

# INVESTMENT DIRECTION
if trend_value > 0:
    investment = "Favorable region/category for expansion"
elif trend_value < 0:
    investment = "High risk of reduced returns, limit investment"
else:
    investment = "Stable market, maintain cautious investment"



## 7. Final report

In [ ]:

print("\n========================================")
print("BUSINESS DECISION REPORT")
print("========================================")

print(f"Overall Outlook        : {outlook}")
print(f"Demand Signal          : {demand}")
print(f"Risk Level             : {risk}")

print("\nRecommended Action:")
print(action)

print("\nInvestment Guidance:")
print(investment)

print("\n3-Month Outlook:")
for date, value in forecast.items():
    print(f"{date.date()} : {value:,.0f} DA")

In [ ]:
#  BEST PRODUCTS (SELECTED WILAYA)

print("\n========================================")
print("PRODUCT STRATEGY")
print("========================================")
print(f"WILAYA: {selected}")

# Filter to selected wilaya
wilaya_data = transactions[transactions["wilaya"] == selected]

# Aggregate product performance
product_summary = (
    wilaya_data
    .groupby("categorie_produit")["montant_da"]
    .sum()
    .reset_index()
)

# Sort by performance
product_summary = product_summary.sort_values(
    "montant_da",
    ascending=False
)

# Top products
top_n = 5
top_products = product_summary.head(top_n)

print("\nTop Products for this Wilaya:\n")

for _, row in top_products.iterrows():
    print(f"- {row['categorie_produit']} : {row['montant_da']:,.0f} DA")

In [ ]:
# ==========================================
# PART 5 - PROFIT vs RISK TRADEOFF
# ==========================================

import numpy as np

print("\n========================================")
print("DECISION TRADEOFF ANALYSIS")
print("========================================")
print(f"WILAYA: {selected}")

# ==========================================
# BASE METRICS
# ==========================================

historical_mean = series.mean()
forecast_mean = forecast.mean()

historical_volatility = series.std() / series.mean()
forecast_volatility = forecast.std() / forecast.mean()

expected_change = (forecast_mean - historical_mean) / historical_mean

# proxy for "profit potential"
# (higher demand → higher revenue opportunity)
profit_potential = forecast_mean

# ==========================================
# STRATEGY SCENARIOS
# ==========================================

scenarios = {
    "SAFE": {
        "risk_multiplier": 0.7,
        "inventory_factor": 0.9
    },
    "BALANCED": {
        "risk_multiplier": 1.0,
        "inventory_factor": 1.0
    },
    "AGGRESSIVE": {
        "risk_multiplier": 1.3,
        "inventory_factor": 1.2
    }
}

# ==========================================
# COMPUTE TRADEOFFS
# ==========================================

print("\nStrategy Comparison:\n")

for name, params in scenarios.items():

    risk_score = (historical_volatility + abs(expected_change)) * params["risk_multiplier"]

    expected_profit = profit_potential * params["inventory_factor"]

    # simple interpretation bands
    if risk_score < 0.25:
        risk_label = "LOW"
    elif risk_score < 0.6:
        risk_label = "MEDIUM"
    else:
        risk_label = "HIGH"

    print("----------------------------------------")
    print(f"Strategy: {name}")
    print(f"Expected Revenue Opportunity: {expected_profit:,.0f} DA")
    print(f"Planning Risk: {risk_label}")
    print(f"Risk Score: {risk_score:.2f}")

# ==========================================
# RECOMMENDED STRATEGY
# ==========================================

best_strategy = min(
    scenarios.keys(),
    key=lambda s: (
        (historical_volatility + abs(expected_change)) * scenarios[s]["risk_multiplier"]
    )
)

print("\n========================================")
print("RECOMMENDATION")
print("========================================")

print(f"Recommended Strategy: {best_strategy}")

if best_strategy == "SAFE":
    print("Focus on stability and avoiding overstock.")
elif best_strategy == "BALANCED":
    print("Maintain standard inventory policy.")
else:
    print("Pursue growth with higher inventory exposure.")